In [ ]:
import numpy as np
import pandas as pd
import os
from pathlib import Path


In [ ]:

def add_cyclical_time_features(df: pd.DataFrame, timestamp_col: str) -> pd.DataFrame:

    ts = pd.to_datetime(df[timestamp_col])

    fractional_hour = ts.dt.hour + (ts.dt.minute / 60.0) + (ts.dt.second / 3600.0)
    radians_24h = 2 * np.pi * (fractional_hour / 24.0)

    df["Hour_X"] = np.cos(radians_24h)
    df["Hour_Y"] = np.sin(radians_24h)

    fractional_dow = ts.dt.dayofweek + (fractional_hour / 24.0)
    radians_dow = 2 * np.pi * (fractional_dow / 7.0)
    df["DoW_X"] = np.cos(radians_dow)
    df["DoW_Y"] = np.sin(radians_dow)

    df["is_weekend"] = (ts.dt.dayofweek >= 5).astype(int)

    return df

In [ ]:
import pandas as pd

REFIT_MAPPINGS = {
    1: {
        "Appliance1": "Fridge",
        "Appliance2": "Chest Freezer",
        "Appliance3": "Upright Freezer",
        "Appliance4": "Tumble Dryer",
        "Appliance5": "Washing Machine",
        "Appliance6": "Dishwasher",
        "Appliance7": "Computer Site",
        "Appliance8": "Television Site",
        "Appliance9": "Electric Heater",
    },
    2: {
        "Appliance1": "Fridge-Freezer",
        "Appliance2": "Washing Machine",
        "Appliance3": "Dishwasher",
        "Appliance4": "Television Site",
        "Appliance5": "Microwave",
        "Appliance6": "Toaster",
        "Appliance7": "Hi-Fi",
        "Appliance8": "Kettle",
        "Appliance9": "Oven Extractor Fan",
    },
    3: {
        "Appliance1": "Toaster",
        "Appliance2": "Fridge-Freezer",
        "Appliance3": "Freezer",
        "Appliance4": "Tumble Dryer",
        "Appliance5": "Dishwasher",
        "Appliance6": "Washing Machine",
        "Appliance7": "Television Site",
        "Appliance8": "Microwave",
        "Appliance9": "Kettle",
    },
    4: {
        "Appliance1": "Fridge",
        "Appliance2": "Freezer",
        "Appliance3": "Fridge-Freezer",
        "Appliance4": "Washing Machine (1)",
        "Appliance5": "Washing Machine (2)",
        "Appliance6": "Computer Site",
        "Appliance7": "Television Site",
        "Appliance8": "Microwave",
        "Appliance9": "Kettle",
    },
    5: {
        "Appliance1": "Fridge-Freezer",
        "Appliance2": "Tumble Dryer",
        "Appliance3": "Washing Machine",
        "Appliance4": "Dishwasher",
        "Appliance5": "Computer Site",
        "Appliance6": "Television Site",
        "Appliance7": "Combination Microwave",
        "Appliance8": "Kettle",
        "Appliance9": "Toaster",
    },
    6: {
        "Appliance1": "Freezer (Utility Room)",
        "Appliance2": "Washing Machine",
        "Appliance3": "Dishwasher",
        "Appliance4": "MJY Computer",
        "Appliance5": "Television Site",
        "Appliance6": "Microwave",
        "Appliance7": "Kettle",
        "Appliance8": "Toaster",
        "Appliance9": "PGM Computer",
    },
    7: {
        "Appliance1": "Fridge",
        "Appliance2": "Freezer (Garage)",
        "Appliance3": "Freezer",
        "Appliance4": "Tumble Dryer",
        "Appliance5": "Washing Machine",
        "Appliance6": "Dishwasher",
        "Appliance7": "Television Site",
        "Appliance8": "Toaster",
        "Appliance9": "Kettle",
    },
    8: {
        "Appliance1": "Fridge",
        "Appliance2": "Freezer",
        "Appliance3": "Dryer",
        "Appliance4": "Washing Machine",
        "Appliance5": "Toaster",
        "Appliance6": "Computer",
        "Appliance7": "Television Site",
        "Appliance8": "Microwave",
        "Appliance9": "Kettle",
    },
    9: {
        "Appliance1": "Fridge-Freezer",
        "Appliance2": "Washer Dryer",
        "Appliance3": "Washing Machine",
        "Appliance4": "Dishwasher",
        "Appliance5": "Television Site",
        "Appliance6": "Microwave",
        "Appliance7": "Kettle",
        "Appliance8": "Hi-Fi",
        "Appliance9": "Electric Heater",
    },
    9: {
        "Appliance1": "Fridge-Freezer",
        "Appliance2": "Washer Dryer",
        "Appliance3": "Washing Machine",
        "Appliance4": "Dishwasher",
        "Appliance5": "Television Site",
        "Appliance6": "Microwave",
        "Appliance7": "Kettle",
        "Appliance8": "Hi-Fi",
        "Appliance9": "Electric Heater",
    },
    10: {
        "Appliance1": "Magimix Blender",
        "Appliance2": "Chest Freezer",
        "Appliance3": "Freezer",
        "Appliance4": "Washing Machine",
        "Appliance5": "Dishwasher",
        "Appliance6": "Television Site",
        "Appliance7": "Microwave",
        "Appliance8": "Toaster",
        "Appliance9": "Hi-Fi",
    },
    11: {
        "Appliance1": "Fridge",
        "Appliance2": "Fridge-Freezer",
        "Appliance3": "Washing Machine",
        "Appliance4": "Dishwasher",
        "Appliance5": "Computer Site",
        "Appliance6": "Microwave",
        "Appliance7": "Kettle",
        "Appliance8": "Router",
        "Appliance9": "Hi-Fi",
    },
    12: {
        "Appliance1": "Fridge-Freezer",
        "Appliance2": "Television Site",
        "Appliance3": "Microwave",
        "Appliance4": "Kettle",
        "Appliance5": "Toaster",
        "Appliance6": "Computer",
        "Appliance7": "Unknown",
        "Appliance8": "Unknown",
        "Appliance9": "Unknown",
    },
    13: {
        "Appliance1": "Television Site",
        "Appliance2": "Dishwasher",
        "Appliance3": "Washing Machine",
        "Appliance4": "Tumble Dryer",
        "Appliance5": "Microwave",
        "Appliance6": "Kettle",
        "Appliance7": "Toaster",
        "Appliance8": "Fridge",
        "Appliance9": "Freezer",
    },
    14: {
        "Appliance1": "Unknown",
        "Appliance2": "Unknown",
        "Appliance3": "Unknown",
        "Appliance4": "Unknown",
        "Appliance5": "Unknown",
        "Appliance6": "Unknown",
        "Appliance7": "Unknown",
        "Appliance8": "Unknown",
        "Appliance9": "Unknown",
    },
    15: {
        "Appliance1": "Fridge-Freezer",
        "Appliance2": "Tumble Dryer",
        "Appliance3": "Washing Machine",
        "Appliance4": "Dishwasher",
        "Appliance5": "Computer Site",
        "Appliance6": "Television Site",
        "Appliance7": "Microwave",
        "Appliance8": "Hi-Fi",
        "Appliance9": "Toaster",
    },
    16: {
        "Appliance1": "Fridge-Freezer (1)",
        "Appliance2": "Fridge-Freezer (2)",
        "Appliance3": "Washing Machine",
        "Appliance4": "Dishwasher",
        "Appliance5": "Computer Site",
        "Appliance6": "Television Site",
        "Appliance7": "Microwave",
        "Appliance8": "Kettle",
        "Appliance9": "Dehumidifier",
    },
    17: {
        "Appliance1": "Freezer",
        "Appliance2": "Fridge-Freezer",
        "Appliance3": "Washing Machine",
        "Appliance4": "Tumble Dryer",
        "Appliance5": "Computer Site",
        "Appliance6": "Television Site",
        "Appliance7": "Microwave",
        "Appliance8": "Kettle",
        "Appliance9": "Unknown",
    },
    18: {
        "Appliance1": "Fridge-Freezer",
        "Appliance2": "Freezer",
        "Appliance3": "Washing Machine",
        "Appliance4": "Dishwasher",
        "Appliance5": "Computer Site",
        "Appliance6": "Television Site",
        "Appliance7": "Microwave",
        "Appliance8": "Kettle",
        "Appliance9": "Toaster",
    },
    19: {
        "Appliance1": "Fridge-Freezer",
        "Appliance2": "Washing Machine",
        "Appliance3": "Television Site",
        "Appliance4": "Microwave",
        "Appliance5": "Kettle",
        "Appliance6": "Toaster",
        "Appliance7": "Breadmaker",
        "Appliance8": "Lamp",
        "Appliance9": "Hi-Fi",
    },
    20: {
        "Appliance1": "Fridge",
        "Appliance2": "Freezer",
        "Appliance3": "Washing Machine",
        "Appliance4": "Tumble Dryer",
        "Appliance5": "Dishwasher",
        "Appliance6": "Computer Site",
        "Appliance7": "Television Site",
        "Appliance8": "Microwave",
        "Appliance9": "Kettle",
    },
    21: {
        "Appliance1": "Fridge-Freezer",
        "Appliance2": "Washing Machine",
        "Appliance3": "Dishwasher",
        "Appliance4": "Television Site",
        "Appliance5": "Microwave",
        "Appliance6": "Kettle",
        "Appliance7": "Toaster",
        "Appliance8": "Vivarium",
        "Appliance9": "Pond Pump",
    },
}


def rename_refit_columns(df: pd.DataFrame, house_id: int) -> pd.DataFrame:

    cols_to_drop = ["Unix", "Issues"]
    df = df.drop(columns=cols_to_drop, errors="ignore")
   
    if house_id not in REFIT_MAPPINGS:
        raise ValueError(f"House {house_id} mapping is not defined.")

    mapping = REFIT_MAPPINGS[house_id]
    print(f"Mapping appliance names for house {house_id}.")

    return df.rename(columns=mapping)




In [ ]:
def process_all_houses(data_dir: str = ".", output_dir: str = "./processed"):
    os.makedirs(output_dir, exist_ok=True)

    for house_id in range(1, 22):
        filename = f"CLEAN_House{house_id}.csv"
        file_path = os.path.join(data_dir, filename)
        if not os.path.isfile(file_path):
            continue

        print(f"Processing House {house_id} from {file_path}...")
        df = pd.read_csv(file_path)
        df = add_cyclical_time_features(df, timestamp_col="Time")
        df = rename_refit_columns(df, house_id=house_id)

        out_path = os.path.join(output_dir, f"House_{house_id}_processed.csv")
        df.to_csv(out_path, index=False)
        print(f"Saved: {out_path}")


process_all_houses(data_dir="./CLEAN_REFIT_081116", output_dir="./processed")


Extracting specific appliance data from the related houses

In [ ]:
REFIT_DIR = Path("./processed")
OUTPUT_DIR = Path("./processed_2")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

APPLIANCE_HOUSES = {
    # "Television Site": [2, 3, 5, 6, 7, 15, 16, 17, 18, 19, 20],
    "Television Site": [2, 3, 5, 6, 7],

    # "Toaster": [2, 3, 5, 6, 7, 12, 15, 19],
    "Toaster": [2, 3, 5, 6, 7],

    # "Microwave": [2, 3, 5, 6, 11, 12, 15, 17, 18, 19, 20],
    "Microwave": [2, 3, 5, 6, 11],

    # "Kettle": [2, 3, 5, 6, 7, 11, 12, 15, 17, 19, 20],
    "Kettle": [2, 3, 5, 6, 7],

    # "Computer Site": [1, 5, 6, 11, 15, 16, 17, 18, 20],
    "Computer Site": [1, 5, 6, 11, 15],

    # "Washing Machine": [1, 2, 3, 5, 6, 7, 11, 15, 16, 17, 18, 19, 20],
    "Washing Machine": [1, 2, 3, 5, 6],

    # "Dishwasher": [1, 2, 3, 5, 6, 7, 11, 15, 16, 18, 20],
    "Dishwasher": [1, 2, 3, 5, 6],
}
TIME_COLUMNS = ["Time", "Hour_X", "Hour_Y", "DoW_X", "DoW_Y", "is_weekend"]
AGGREGATE_COLUMN = "Aggregate"
APPLIANCES = ["Television Site", "Toaster", "Microwave", "Kettle", "Computer Site", "Washing Machine", "Dishwasher"]

In [ ]:
def load_house_data(house_number):
    path_to_house_data = REFIT_DIR / f"House_{house_number}_processed.csv"
    
    if path_to_house_data is None:
        raise FileNotFoundError(
            f"Could not find data file for House {house_number}.\n"
        )

    print(f"Loading House {house_number}: {path_to_house_data}")

    df = pd.read_csv(path_to_house_data)

    print(f"    Rows: {len(df):,}")
    print(f"    Columns: {list(df.columns)}")

    return df

In [ ]:
def extract_appliance_data(df, house_number, appliance):
    """ Extract: Time related columns | Aggregate | Target appliance
    """
    result = df[["Time", "Hour_X", "Hour_Y", "DoW_X", "DoW_Y", "is_weekend", "Aggregate", appliance]].copy()
    result["house"] = house_number
    result["Time"] = pd.to_datetime(result["Time"])
    result = result.dropna()
    return result

In [ ]:
def prepare_appliance_dataset(appliance, house_list):
    print("\n" + "-" * 70)
    print(f"Preparing: {appliance}")
    print(f"Houses: {house_list}")
    print("-" * 70)

    house_data = []
    for house in house_list:
        try:
            df = load_house_data(house)
            # print(df.columns)
            extracted = extract_appliance_data(df, house, appliance)
            house_data.append(extracted)
            print(f"--------- Extracted {len(extracted):,} valid rows"
            )
        except Exception as e:
            print(f"\nERROR processing House {house} for {appliance}:")
            print(e)

    if not house_data:
        raise RuntimeError(f"No data found for appliance '{appliance}'.")

    combined = pd.concat(house_data, ignore_index=True)
    combined = combined.sort_values(["house", "Time"])

    # Remove duplicates
    combined = combined.drop_duplicates(subset=["house", "Time"])
    # Reset index
    combined = combined.reset_index(drop=True)

    output_file = OUTPUT_DIR / f"{appliance}_prepared.csv"
    combined.to_csv(output_file, index=False)

    print("\nFinal dataset:")
    print(f"    Appliance: {appliance}")
    print(f"    Houses: {combined['house'].unique().tolist()}")
    print(f"    Rows: {len(combined):,}")
    print(f"    Saved: {output_file}")

    return combined

In [ ]:
prepared_data = {}
for appliance, houses in APPLIANCE_HOUSES.items():
    data = prepare_appliance_dataset(appliance, houses)
    prepared_data[appliance] = data


In [ ]:
import os
import pandas as pd

# http://arxiv.org/pdf/1507.06594
ON_POWER_THRESHOLDS = {
    "Television Site": 15.0,
    "Kettle": 2000.0,
    "Microwave": 200.0,
    "Toaster": 1000.0,
    "Washing Machine": 20.0,
    "Dishwasher": 10.0,
    "Computer Site": 20.0,
}    


def add_binary_activation_column(file_path: str, appliance_name: str, output_path: str = None,) -> pd.DataFrame:
    print(f"\nLooking for the Threshold of the Current Appliance: {appliance_name}\n")
    threshold = ON_POWER_THRESHOLDS[appliance_name]
    df = pd.read_csv(file_path)

    target_col = appliance_name
    status_col_name = f"{appliance_name}_is_on"
    df[status_col_name] = (df[target_col] >= threshold).astype(int).where(df[target_col].notna(), other=float("nan"))

    if output_path:
        os.makedirs(os.path.dirname(output_path), exist_ok=True)
        df.to_csv(output_path, index=False)
        print(
            f"Saved: {output_path} (Threshold: {threshold} W applied to '{target_col}')"
        )

    return df


def add_binary_activation_to_all(input_dir: str, output_dir: str):
    os.makedirs(output_dir, exist_ok=True)

    for filename in os.listdir(input_dir):
        appliance_name = os.path.splitext(filename)[0]
        print(f"\nCurrent Appliance: {appliance_name}\n")
        input_path = os.path.join(input_dir, filename)
        output_path = os.path.join(output_dir, f"{appliance_name}_activation_labeled.csv")
        add_binary_activation_column(file_path=input_path, appliance_name=appliance_name, output_path=output_path)



In [ ]:
add_binary_activation_to_all(input_dir="./processed_2", 
                             output_dir="./REFIT_binary_activation_labeled")

Combining all appliance data

In [ ]:
DATA_DIR = Path("./REFIT_binary_activation_labeled")
OUTPUT_FILE = DATA_DIR / "REFIT_multi_appliance_binary.csv"

APPLIANCE_FILES = {
    "Television Site": "Television Site.csv",
    "Toaster": "Toaster.csv",
    "Microwave": "Microwave.csv",
    "Kettle": "Kettle.csv",
    "Computer Site": "Computer Site.csv",
    "Washing Machine": "Washing Machine.csv",
    "Dishwasher": "Dishwasher.csv",
}

In [ ]:
datasets = []

for appliance, filename in APPLIANCE_FILES.items():

    file_path = DATA_DIR / filename
    if not file_path.exists():
        raise FileNotFoundError(f"Could not find: {file_path}")
    
    print(f"\nLoading {appliance}: {file_path}")

    df = pd.read_csv(file_path)
    df["Time"] = pd.to_datetime(df["Time"])
    df = df.dropna(subset=["Time"])
    df = df.dropna(subset=["house"])
    df["house"] = df["house"].astype(int)
    print(f"Rows: {len(df):,}")

    print(f"Houses: {sorted(df['house'].unique().tolist())}")

    datasets.append((appliance, df))

In [ ]:
_, merged = datasets[0]
for appliance, df in datasets[1:]:
    df_to_merge = df.copy()
    merged = merged.merge(df_to_merge, on=["Time", "Hour_X", "Hour_Y", "DoW_X", "DoW_Y", "is_weekend", "Aggregate", "house"], how="outer")
    
merged = merged.sort_values(["house", "Time"])
merged = merged.reset_index(drop=True)

In [ ]:
merged.head()

In [ ]:
merged.to_csv(OUTPUT_FILE, index=False)